# Temporal Cross-Validation for Medical Time Series[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ki-smile/trustcv/blob/main/notebooks/03_Temporal_Medical.ipynb)## 🎯 Learning Objectives- Understand temporal dependencies in medical data- Implement time-aware cross-validation strategies- Handle ICU monitoring and disease progression data- Avoid temporal leakage (future data in training)- Apply rolling window and expanding window validation- Implement blocked and purged time series CV---

## 1. The Challenge: Time Matters in MedicineMedical data often has strong temporal components:- **Disease Progression**: Conditions evolve over time- **Treatment Effects**: Interventions have delayed impacts- **Seasonal Patterns**: Flu seasons, allergy cycles- **ICU Monitoring**: Continuous vital signs- **Clinical Trials**: Longitudinal patient follow-ups**Random splitting violates temporal order and causes:**- ⚠️ Future information leaking into training (temporal leakage)- ⚠️ Overly optimistic performance estimates- ⚠️ Models that fail in real-time deployment

In [ ]:
# Setupimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom datetime import datetime, timedeltaimport warningswarnings.filterwarnings('ignore')# ML importsfrom trustcv import TimeSeriesSplit, cross_val_scorefrom sklearn.ensemble import RandomForestRegressor, GradientBoostingClassifierfrom sklearn.metrics import mean_absolute_error, roc_auc_scorefrom sklearn.preprocessing import StandardScaler# Visualization settingsplt.style.use('seaborn-v0_8-whitegrid')colors = ['#870052', '#FF876F', '#4F0433', '#EDF4F4']sns.set_palette(colors)print("✅ Libraries loaded successfully!")

## 2. Generating Realistic Medical Time Series DataLet's simulate three common medical time series scenarios:1. **ICU Patient Monitoring** - High-frequency vital signs2. **Disease Progression** - Monthly clinical measurements3. **Epidemic Surveillance** - Daily case counts

In [ ]:
def generate_icu_monitoring_data(n_patients=50, hours_per_patient=168):    """    Generate ICU monitoring data with vital signs    Simulates 1 week of hourly measurements    """    np.random.seed(42)    data = []        base_date = pd.Timestamp('2024-01-01')        for patient_id in range(n_patients):        # Patient baseline characteristics        age = np.random.normal(65, 15)        severity = np.random.random()  # Illness severity                # Admission time (staggered admissions)        admission = base_date + timedelta(days=np.random.randint(0, 30))                # Generate hourly measurements        for hour in range(hours_per_patient):            timestamp = admission + timedelta(hours=hour)                        # Vital signs with temporal correlation            # Add circadian rhythm and random walk            circadian = np.sin(2 * np.pi * hour / 24)            trend = severity * hour / 168  # Worsening trend for severe cases                        heart_rate = 70 + 20 * severity + 5 * circadian + np.random.normal(0, 5)            blood_pressure = 120 - 30 * severity - 5 * circadian + np.random.normal(0, 10)            respiratory_rate = 16 + 8 * severity + 2 * circadian + np.random.normal(0, 2)            temperature = 37 + 1.5 * severity + 0.3 * circadian + np.random.normal(0, 0.3)            oxygen_sat = 98 - 10 * severity - 2 * circadian + np.random.normal(0, 2)                        # Deterioration event (binary outcome)            deterioration_risk = severity * 0.3 + trend * 0.5 + (temperature > 38.5) * 0.2            deterioration = 1 if np.random.random() < deterioration_risk else 0                        data.append({                'patient_id': f'ICU_{patient_id:03d}',                'timestamp': timestamp,                'hour_since_admission': hour,                'age': age,                'heart_rate': heart_rate,                'blood_pressure': blood_pressure,                'respiratory_rate': respiratory_rate,                'temperature': temperature,                'oxygen_saturation': oxygen_sat,                'deterioration_next_6h': deterioration            })        return pd.DataFrame(data)# Generate ICU dataicu_df = generate_icu_monitoring_data()print(f"📊 ICU Monitoring Dataset:")print(f"  - Shape: {icu_df.shape}")print(f"  - Patients: {icu_df['patient_id'].nunique()}")print(f"  - Time span: {icu_df['timestamp'].min()} to {icu_df['timestamp'].max()}")print(f"  - Deterioration rate: {icu_df['deterioration_next_6h'].mean():.1%}\n")icu_df.head()

In [ ]:
def generate_disease_progression_data(n_patients=200, n_months=24):    """    Generate chronic disease progression data    Monthly measurements over 2 years    """    np.random.seed(42)    data = []        base_date = pd.Timestamp('2022-01-01')        for patient_id in range(n_patients):        # Patient characteristics        baseline_severity = np.random.beta(2, 5)  # Skewed towards mild        progression_rate = np.random.exponential(0.02)        treatment_response = np.random.random()                # Monthly follow-ups        for month in range(n_months):            timestamp = base_date + pd.DateOffset(months=month)                        # Disease markers with progression            natural_progression = baseline_severity * (1 + progression_rate * month)            treatment_effect = -treatment_response * month * 0.01 if month > 6 else 0            seasonal_effect = 0.1 * np.sin(2 * np.pi * month / 12)                        biomarker1 = natural_progression + treatment_effect + seasonal_effect + np.random.normal(0, 0.1)            biomarker2 = biomarker1 * 1.5 + np.random.normal(0, 0.2)            symptom_score = np.clip(biomarker1 * 10, 0, 10)            quality_of_life = np.clip(10 - biomarker1 * 8, 0, 10)                        # Hospitalization risk            hosp_risk = biomarker1 * 0.3 + (symptom_score > 7) * 0.4            hospitalized = 1 if np.random.random() < hosp_risk else 0                        data.append({                'patient_id': f'PT_{patient_id:04d}',                'timestamp': timestamp,                'months_since_diagnosis': month,                'biomarker1': biomarker1,                'biomarker2': biomarker2,                'symptom_score': symptom_score,                'quality_of_life': quality_of_life,                'on_treatment': 1 if month > 6 else 0,                'hospitalized': hospitalized            })        return pd.DataFrame(data)# Generate disease progression datadisease_df = generate_disease_progression_data()print(f"📊 Disease Progression Dataset:")print(f"  - Shape: {disease_df.shape}")print(f"  - Patients: {disease_df['patient_id'].nunique()}")print(f"  - Time span: {disease_df['timestamp'].min()} to {disease_df['timestamp'].max()}")print(f"  - Hospitalization rate: {disease_df['hospitalized'].mean():.1%}\n")disease_df.head()

## 3. Visualizing Temporal Patterns

In [ ]:
# Visualize temporal patternsfig, axes = plt.subplots(2, 2, figsize=(14, 10))# ICU: Vital signs over time for sample patientssample_patients = icu_df['patient_id'].unique()[:3]for pid in sample_patients:    patient_data = icu_df[icu_df['patient_id'] == pid]    axes[0, 0].plot(patient_data['hour_since_admission'],                     patient_data['heart_rate'],                     label=pid, alpha=0.7)axes[0, 0].set_xlabel('Hours Since Admission')axes[0, 0].set_ylabel('Heart Rate (bpm)')axes[0, 0].set_title('ICU: Heart Rate Monitoring')axes[0, 0].legend()axes[0, 0].grid(True, alpha=0.3)# ICU: Deterioration events over timehourly_deterioration = icu_df.groupby('hour_since_admission')['deterioration_next_6h'].mean()axes[0, 1].plot(hourly_deterioration.index, hourly_deterioration.values,                 color=colors[1], linewidth=2)axes[0, 1].fill_between(hourly_deterioration.index, hourly_deterioration.values,                         alpha=0.3, color=colors[1])axes[0, 1].set_xlabel('Hours Since Admission')axes[0, 1].set_ylabel('Deterioration Rate')axes[0, 1].set_title('ICU: Deterioration Risk Over Time')axes[0, 1].grid(True, alpha=0.3)# Disease Progression: Biomarker evolutionsample_patients_disease = disease_df['patient_id'].unique()[:5]for pid in sample_patients_disease:    patient_data = disease_df[disease_df['patient_id'] == pid]    axes[1, 0].plot(patient_data['months_since_diagnosis'],                     patient_data['biomarker1'],                     alpha=0.5)axes[1, 0].axvline(x=6, color='red', linestyle='--', alpha=0.5, label='Treatment Start')axes[1, 0].set_xlabel('Months Since Diagnosis')axes[1, 0].set_ylabel('Biomarker Level')axes[1, 0].set_title('Disease Progression: Biomarker Evolution')axes[1, 0].legend()axes[1, 0].grid(True, alpha=0.3)# Monthly hospitalization ratemonthly_hosp = disease_df.groupby('months_since_diagnosis')['hospitalized'].mean()axes[1, 1].bar(monthly_hosp.index, monthly_hosp.values,                color=colors[2], alpha=0.7)axes[1, 1].set_xlabel('Months Since Diagnosis')axes[1, 1].set_ylabel('Hospitalization Rate')axes[1, 1].set_title('Monthly Hospitalization Risk')axes[1, 1].grid(True, alpha=0.3)plt.tight_layout()plt.show()print("📈 Key Observations:")print("- ICU: Clear circadian patterns in vital signs")print("- ICU: Deterioration risk increases with length of stay")print("- Disease: Treatment effect visible after month 6")print("- Disease: Hospitalization risk varies over time")

## 4. The Problem with Random Splitting on Time Series

In [ ]:
# Demonstrate temporal leakage with random splitfrom sklearn.model_selection import train_test_split# Prepare features and target for disease progressionfeature_cols = ['months_since_diagnosis', 'biomarker1', 'biomarker2',                 'symptom_score', 'on_treatment']X = disease_df[feature_cols].valuesy = disease_df['hospitalized'].valuestimestamps = disease_df['timestamp'].values# Method 1: Random split (WRONG for time series!)X_train_random, X_test_random, y_train_random, y_test_random, time_train, time_test = train_test_split(    X, y, timestamps, test_size=0.2, random_state=42)print("❌ RANDOM SPLIT (Wrong for temporal data):")print(f"  Train period: {pd.to_datetime(time_train).min()} to {pd.to_datetime(time_train).max()}")print(f"  Test period:  {pd.to_datetime(time_test).min()} to {pd.to_datetime(time_test).max()}")print(f"  ⚠️ Test data BEFORE training data - TEMPORAL LEAKAGE!\n")# Method 2: Temporal split (CORRECT!)split_date = disease_df['timestamp'].quantile(0.8)train_mask = disease_df['timestamp'] < split_datetest_mask = disease_df['timestamp'] >= split_dateX_train_temporal = X[train_mask]X_test_temporal = X[test_mask]y_train_temporal = y[train_mask]y_test_temporal = y[test_mask]print("✅ TEMPORAL SPLIT (Correct):")print(f"  Train period: {disease_df[train_mask]['timestamp'].min()} to {disease_df[train_mask]['timestamp'].max()}")print(f"  Test period:  {disease_df[test_mask]['timestamp'].min()} to {disease_df[test_mask]['timestamp'].max()}")print(f"  ✓ Training always before test - NO LEAKAGE!")

In [ ]:
# Compare model performancefrom sklearn.metrics import accuracy_score, roc_auc_score# Train modelsclf_random = GradientBoostingClassifier(n_estimators=100, random_state=42)clf_temporal = GradientBoostingClassifier(n_estimators=100, random_state=42)clf_random.fit(X_train_random, y_train_random)clf_temporal.fit(X_train_temporal, y_train_temporal)# Evaluateresults = pd.DataFrame({    'Method': ['Random Split (Leakage)', 'Temporal Split (No Leakage)'],    'Train Accuracy': [        accuracy_score(y_train_random, clf_random.predict(X_train_random)),        accuracy_score(y_train_temporal, clf_temporal.predict(X_train_temporal))    ],    'Test Accuracy': [        accuracy_score(y_test_random, clf_random.predict(X_test_random)),        accuracy_score(y_test_temporal, clf_temporal.predict(X_test_temporal))    ],    'Test AUC': [        roc_auc_score(y_test_random, clf_random.predict_proba(X_test_random)[:, 1]),        roc_auc_score(y_test_temporal, clf_temporal.predict_proba(X_test_temporal)[:, 1])    ]})print("\n📊 Performance Comparison:")print(results.to_string(index=False))print("\n⚠️ Random split shows inflated performance due to temporal leakage!")

## 5. Time Series Cross-Validation Methods

In [ ]:
# Implement different time series CV methodsfrom sklearn.model_selection import TimeSeriesSplit# Prepare data sorted by timedisease_sorted = disease_df.sort_values('timestamp').reset_index(drop=True)X_sorted = disease_sorted[feature_cols].valuesy_sorted = disease_sorted['hospitalized'].values# Visualize different CV strategiesfig, axes = plt.subplots(4, 1, figsize=(14, 12))cv_methods = [    ('Time Series Split (Expanding Window)', TimeSeriesSplit(n_splits=5)),    ('Rolling Window (Fixed Size)', None),  # Custom implementation    ('Blocked Time Series', None),  # Custom implementation    ('Purged & Embargoed', None)  # Custom implementation]# 1. Standard TimeSeriesSplittscv = TimeSeriesSplit(n_splits=5)for fold_idx, (train_idx, test_idx) in enumerate(tscv.split(X_sorted)):    train_period = np.zeros(len(X_sorted))    train_period[train_idx] = 1    test_period = np.zeros(len(X_sorted))    test_period[test_idx] = 1        axes[0].fill_between(range(len(X_sorted)),                         fold_idx + train_period * 0.8,                         fold_idx, alpha=0.3, color=colors[0], label='Train' if fold_idx == 0 else '')    axes[0].fill_between(range(len(X_sorted)),                         fold_idx + test_period * 0.8,                         fold_idx, alpha=0.7, color=colors[1], label='Test' if fold_idx == 0 else '')axes[0].set_title('Time Series Split (Expanding Window)')axes[0].set_ylabel('Fold')axes[0].legend()axes[0].set_ylim(-0.5, 5.5)# 2. Rolling Window (Fixed training size)window_size = len(X_sorted) // 3step_size = len(X_sorted) // 6for fold_idx in range(5):    train_start = fold_idx * step_size    train_end = train_start + window_size    test_start = train_end    test_end = min(test_start + step_size, len(X_sorted))        if test_end > len(X_sorted):        break        train_period = np.zeros(len(X_sorted))    train_period[train_start:train_end] = 1    test_period = np.zeros(len(X_sorted))    test_period[test_start:test_end] = 1        axes[1].fill_between(range(len(X_sorted)),                         fold_idx + train_period * 0.8,                         fold_idx, alpha=0.3, color=colors[0])    axes[1].fill_between(range(len(X_sorted)),                         fold_idx + test_period * 0.8,                         fold_idx, alpha=0.7, color=colors[1])axes[1].set_title('Rolling Window (Fixed Training Size)')axes[1].set_ylabel('Fold')axes[1].set_ylim(-0.5, 5.5)# 3. Blocked Time Series (Contiguous blocks)block_size = len(X_sorted) // 6for fold_idx in range(5):    test_start = fold_idx * block_size    test_end = test_start + block_size        train_mask = np.ones(len(X_sorted), dtype=bool)    train_mask[test_start:test_end] = False        train_period = train_mask.astype(float)    test_period = (~train_mask).astype(float)        axes[2].fill_between(range(len(X_sorted)),                         fold_idx + train_period * 0.8,                         fold_idx, alpha=0.3, color=colors[0])    axes[2].fill_between(range(len(X_sorted)),                         fold_idx + test_period * 0.8,                         fold_idx, alpha=0.7, color=colors[1])axes[2].set_title('Blocked Time Series (Non-overlapping Test Blocks)')axes[2].set_ylabel('Fold')axes[2].set_ylim(-0.5, 5.5)# 4. Purged & Embargoed (Gap between train and test)purge_gap = len(X_sorted) // 20  # 5% gapfor fold_idx, (train_idx, test_idx) in enumerate(tscv.split(X_sorted)):    # Add purge gap    train_idx = train_idx[train_idx < test_idx[0] - purge_gap]        train_period = np.zeros(len(X_sorted))    train_period[train_idx] = 1    test_period = np.zeros(len(X_sorted))    test_period[test_idx] = 1    gap_period = np.zeros(len(X_sorted))    if len(train_idx) > 0:        gap_period[train_idx[-1]+1:test_idx[0]] = 1        axes[3].fill_between(range(len(X_sorted)),                         fold_idx + train_period * 0.8,                         fold_idx, alpha=0.3, color=colors[0])    axes[3].fill_between(range(len(X_sorted)),                         fold_idx + gap_period * 0.8,                         fold_idx, alpha=0.3, color='grey', label='Gap' if fold_idx == 0 else '')    axes[3].fill_between(range(len(X_sorted)),                         fold_idx + test_period * 0.8,                         fold_idx, alpha=0.7, color=colors[1])axes[3].set_title('Purged & Embargoed (Gap Between Train and Test)')axes[3].set_ylabel('Fold')axes[3].set_xlabel('Time Index')axes[3].legend()axes[3].set_ylim(-0.5, 5.5)plt.tight_layout()plt.show()print("📊 Time Series CV Methods:")print("1. Expanding Window: Training set grows over time")print("2. Rolling Window: Fixed training size, slides forward")print("3. Blocked: Test on contiguous time blocks")print("4. Purged: Gap prevents information leakage")

## 6. Using trustcv for Temporal Validation

In [ ]:
# Import trustcv temporal splittersfrom trustcv.splitters import TemporalClinical, BlockedTimeSeriesfrom trustcv import MedicalValidator# Temporal validation for ICU dataprint("🏥 ICU Monitoring - Predict Deterioration\n")# Prepare ICU featuresicu_features = ['heart_rate', 'blood_pressure', 'respiratory_rate',                 'temperature', 'oxygen_saturation']X_icu = icu_df[icu_features].valuesy_icu = icu_df['deterioration_next_6h'].valuestimestamps_icu = icu_df['timestamp'].values# Use TemporalClinical splitter with gap (6 hours)temporal_cv = TemporalClinical(n_splits=5, gap=6)# Validate modelvalidator = MedicalValidator(method='temporal', n_splits=5)print("Running temporal cross-validation...")print("Gap: 6 hours between train and test (prediction horizon)\n")# Simple evaluationfrom sklearn.ensemble import RandomForestClassifierscores = []for fold, (train_idx, test_idx) in enumerate(temporal_cv.split(X_icu, timestamps=timestamps_icu)):    X_train, X_test = X_icu[train_idx], X_icu[test_idx]    y_train, y_test = y_icu[train_idx], y_icu[test_idx]        clf = RandomForestClassifier(n_estimators=50, random_state=42)    clf.fit(X_train, y_train)        y_pred_proba = clf.predict_proba(X_test)[:, 1]    auc = roc_auc_score(y_test, y_pred_proba)    scores.append(auc)        print(f"Fold {fold+1}: AUC = {auc:.3f}")print(f"\nMean AUC: {np.mean(scores):.3f} ± {np.std(scores):.3f}")print("\n✅ Temporal validation ensures:")print("  - No future data in training")print("  - Realistic prediction scenario")print("  - 6-hour gap mimics real prediction horizon")

## 7. Advanced: Patient-Specific Temporal CVCombining patient grouping with temporal ordering:

In [ ]:
# Patient-specific temporal validationfrom trustcv.splitters import PurgedGroupTimeSeriesSplitprint("🏥 Disease Progression - Patient-Aware Temporal CV\n")# This ensures:# 1. Temporal order is preserved# 2. Same patient never in train and test# 3. Gap between train and test periods# Prepare dataX_disease = disease_df[feature_cols].valuesy_disease = disease_df['hospitalized'].valuespatient_ids = disease_df['patient_id'].valuestimestamps = disease_df['timestamp'].values# Purged Group Time Series Splitpgts = PurgedGroupTimeSeriesSplit(n_splits=5, purge_gap=30)  # 30-day gapprint("Configuration:")print("  - 5 temporal folds")print("  - 30-day purge gap")print("  - Patient grouping enforced\n")# Evaluatescores_pgts = []patient_overlap_checks = []for fold, (train_idx, test_idx) in enumerate(pgts.split(X_disease, groups=patient_ids, timestamps=timestamps)):    if len(train_idx) == 0 or len(test_idx) == 0:        continue            X_train, X_test = X_disease[train_idx], X_disease[test_idx]    y_train, y_test = y_disease[train_idx], y_disease[test_idx]        # Check patient overlap    train_patients = set(patient_ids[train_idx])    test_patients = set(patient_ids[test_idx])    overlap = train_patients.intersection(test_patients)    patient_overlap_checks.append(len(overlap) == 0)        # Train and evaluate    clf = GradientBoostingClassifier(n_estimators=50, random_state=42)    clf.fit(X_train, y_train)        y_pred_proba = clf.predict_proba(X_test)[:, 1]    auc = roc_auc_score(y_test, y_pred_proba)    scores_pgts.append(auc)        print(f"Fold {fold+1}: AUC = {auc:.3f}, Patient Overlap = {len(overlap)}")print(f"\nMean AUC: {np.mean(scores_pgts):.3f} ± {np.std(scores_pgts):.3f}")print(f"All folds passed patient separation check: {all(patient_overlap_checks)}")print("\n✅ Combined patient-temporal validation ensures:")print("  - No patient in both train and test")print("  - Temporal order preserved")print("  - Purge gap prevents leakage")

## 8. Practical Guidelines for Medical Time Series### When to Use Each Method:| Method | Use Case | Example ||--------|----------|----------|| **TimeSeriesSplit** | Single continuous time series | Hospital admission rates || **Rolling Window** | Stationary processes, fixed context | Vital sign monitoring || **Blocked Time Series** | Seasonal patterns | Flu prediction || **Purged & Embargoed** | Prediction with lag | Disease onset prediction || **Patient Temporal** | Longitudinal patient data | Clinical trials |### Key Considerations:1. **Prediction Horizon**: Match gap to real-world lag2. **Seasonality**: Use blocked CV for seasonal data3. **Non-stationarity**: Consider detrending4. **Sample Size**: Ensure adequate data in each fold5. **Patient Effects**: Combine temporal + grouped CV

In [ ]:
# Comparison of all temporal methodsmethods_comparison = {    'Method': [        'Random Split (Wrong)',        'Simple Temporal Split',        'Time Series CV',        'Rolling Window',        'Purged & Embargoed',        'Patient-Temporal'    ],    'Temporal Order': ['❌', '✅', '✅', '✅', '✅', '✅'],    'No Leakage': ['❌', '✅', '✅', '✅', '✅', '✅'],    'Patient Grouping': ['❌', '❌', '❌', '❌', '❌', '✅'],    'Handles Gaps': ['❌', '❌', '❌', '❌', '✅', '✅'],    'Use Case': [        'Never use for time series',        'Simple forecasting',        'Growing dataset',        'Stationary series',        'Prediction with lag',        'Longitudinal studies'    ]}comparison_df = pd.DataFrame(methods_comparison)print("\n📊 Temporal Cross-Validation Methods Comparison:")print(comparison_df.to_string(index=False))

## 9. Exercise: Implement Custom Temporal CV

In [ ]:
# Exercise 1: Implement day-forward chainingdef day_forward_chaining_cv(df, n_splits=5, min_train_days=30):    """    TODO: Implement forward chaining CV based on days    - Each fold adds more days to training    - Test on next period    - Ensure minimum training days    """    # Your code here    pass# Exercise 2: Implement seasonal CVdef seasonal_cv(df, date_col, season_col=None):    """    TODO: Create CV folds based on seasons    - Train on some seasons, test on others    - Account for year-over-year patterns    """    # Your code here    pass# Exercise 3: Implement sliding window with overlapdef sliding_window_cv(X, y, window_size, step_size, test_size):    """    TODO: Create overlapping sliding windows    - Fixed window size    - Configurable step size    - Fixed test size    """    # Your code here    passprint("📝 Complete the exercises to practice temporal CV implementation!")

## 🎓 Key Takeaways1. **Temporal order is sacred** - Never use future data for training2. **Random splits cause leakage** - Always use time-aware CV3. **Gap = Prediction horizon** - Match your validation to deployment4. **Combine with patient grouping** - For longitudinal studies5. **Consider seasonality** - Use blocked CV when appropriate6. **Monitor distribution shift** - Check if patterns change over time### ✅ Best Practices:- Always sort by time before splitting- Use purge gaps for realistic evaluation- Check for temporal trends and seasonality- Combine temporal + patient grouping when needed- Match CV to your deployment scenario### ❌ Common Mistakes:- Using random splits on time series- Ignoring temporal dependencies- Not accounting for prediction lag- Mixing past and future in features- Using future patient outcomes---### 🚀 Next StepsContinue with:- **[04_Nested_CV.ipynb](04_Nested_CV.ipynb)** - Hyperparameter tuning without leakage- **[Advanced_Methods.ipynb](Advanced_Methods.ipynb)** - Spatial and hierarchical CV---*Created with trustcv - Trustworthy Cross-Validation Toolkit*